# VLM

Nesse notebook, vamos explorar como VLM's podem ser usados para extrair ou processar informações de documentos/ imagens.

In [ ]:
from IPython import get_ipython
if 'google.colab' in str(get_ipython()):
    print("Preparando ambiente Google Colab")
    !pip install opencv-python==5.0.0.93 
    !pip install opencv-contrib-python==5.0.0.93
    !pip install openai
    !git clone https://github.com/pvoloshyn/curso-visao-computacional.git
    %cd curso-visao-computacional
else:
    pass

## Carregando bibliotecas

Além das bibliotecas que usamos em outros notebooks, vamos utilizar mais algumas.
* `openai`: Biblioteca que vamos usar para fazer chamadas aos LLM's da OpenRouter.

In [ ]:
from getpass import getpass
from pathlib import Path
import json
import base64
import mimetypes

from openai import OpenAI

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
# Indica ao notebook to render figures in-page.
%matplotlib inline  
from IPython.display import Image, Markdown

## 1. Obtendo Chave da OpenRouter

O **OpenRouter** é uma plataforma que funciona como um **hub de modelos de IA**. Em vez de integrar seu código separadamente com OpenAI, Google, Anthropic, DeepSeek, Qwen, NVIDIA e outros provedores, você utiliza uma única API e escolhe o modelo que deseja usar.

Para o desenvolvedor, a experiência é muito parecida com a API da OpenAI (por isso usaremos a biblioteca `openai`).

### 1.1. Configurando conta na OpenRouter e obtendo chave

A configuração da conta é bem simples:

1. Entre no site da [OpenRouter](https://openrouter.ai/)
2. Clique em **Sign Up** e preencha as informações (cartão de crédito é opcional e pode ser configurado depois, caso queira).
3. Acesse as [configurações de chaves](https://openrouter.ai/workspaces/default/keys) e clique em **+ New Key**.
4. Dê um nome para a chave e configure como queira (podendo deixar como está).
5. Copie e chave, pois usaremos daqui a pouco. Guarde-a em local seguro.

**OBSERVAÇÕES:**
- Existem diversos modelos gratuitos disponíveis, mas tome cuidado, pois muitos deles utilizam seus dados para treino posterior. Em produção, ative um modelo pago para evitar dor de cabeça.
 

In [ ]:
API_KEY = getpass("Digite a API KEY do OpenRouter")
OPEN_ROUTER_DEFAULT_MODEL = 'google/gemma-4-26b-a4b-it:free'

client = OpenAI(
    api_key=API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

### 1.2. Testando acesso

Vamos fazer um simples teste de uso do modelo.

In [ ]:
def perguntar(
    prompt: str,
    model: str = None,
    temperature: float = 0.2
):
    response = client.chat.completions.create(
        model=model or OPEN_ROUTER_DEFAULT_MODEL,
        temperature=temperature,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.choices[0].message.content

In [ ]:
perguntar("olá, como vai?")

Agora testando com imagens.

In [ ]:
def image_to_data_url(image_path: str) -> str:
    """
    Converte uma imagem para Data URL compatível com OpenAI/OpenRouter.

    Suporta:
        - jpg
        - jpeg
        - png
        - webp
    """
    image_path = Path(image_path)

    if not image_path.exists():
        raise FileNotFoundError(
            f"Arquivo não encontrado: {image_path}"
        )

    mime_type, _ = mimetypes.guess_type(image_path)

    if mime_type not in {
        "image/jpeg",
        "image/png",
        "image/webp",
    }:
        raise ValueError(
            f"Formato não suportado: {mime_type}"
        )

    with open(image_path, "rb") as f:
        encoded = base64.b64encode(
            f.read()
        ).decode("utf-8")

    return f"data:{mime_type};base64,{encoded}"


def perguntar_imagem(
    image_path: str,
    prompt: str,
    model: str = None
):

    response = client.chat.completions.create(
        model=model or OPEN_ROUTER_DEFAULT_MODEL,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": prompt
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": image_to_data_url(
                                image_path
                            )
                        }
                    }
                ]
            }
        ]
    )

    return response.choices[0].message.content

In [ ]:
resposta = perguntar_imagem(
    'imagens/03/teste-ocr.webp',
    'O que está escrito nessa imagem?'
)

Markdown(resposta)

## 2. Processando Documento Simples

Vimos que um IDP segue uma série de passos para se obter dados estruturados de um documento.

> **Input** > OCR > Classificação (opcional) > Extração > Validação > **Resposta**

No caso de um VLM, podemos simplesmente usá-lo para processar todos esses passos em uma única etapa.

> **Input** > VLM > **Resposta**

Vamos utilizar os mesmos documentos e *schemas* do notebook anterior para poder comparar.

In [ ]:
img_bgr = cv2.imread('imagens/03/nota-fiscal.png')
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12, 8))
plt.imshow(img_rgb)
plt.axis('off')
plt.show()

### 2.1. Definindo schema de saída

Vamos utilizar o mesmo schema usado no notebook anterior para essa mesma nota.

In [ ]:
schema = {
    "type": "object",
    "properties": {

        "titulo_documento": {
            "type": "string",
            "description": "Título principal do documento"
        },

        "empresa": {
            "type": "object",
            "properties": {
                "nome": {
                    "type": "string"
                },
                
                "identificacao": {
                    "type": "string"
                }
            }
        },

        "informacoes_gerais": {
            "type": "object",
            "properties": {

                "numero_documento": {
                    "type": "string"
                },

                "data_emissao": {
                    "type": "string",
                    "description": "Formato yyyy-mm-dd"
                },

                "cliente": {
                    "type": "string"
                },

                "destinacao": {
                    "type": "string"
                }
            }
        },

        "itens": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {

                    "codigo": {
                        "type": "string"
                    },

                    "descricao": {
                        "type": "string"
                    },

                    "quantidade": {
                        "type": "number"
                    },

                    "valor_unitario": {
                        "type": "number"
                    },

                    "valor_total": {
                        "type": "number"
                    }

                }
            }
        },
    }
}

### 2.2. Definindo prompt

VLM's, assim como LLM's, precisam receber um prompt para indicar o que se espera extrair. É aqui que podemos definir etapas de classificação, extração, validação, etc.

In [ ]:
prompt = f"""\
Com a nota fiscal fornecida, extraia as seguintes informações em JSON usando a especificação abaixo delimitada entre ###.
###
{json.dumps(schema)}
###

Retorne apenas o JSON.
"""

### 2.3. Processando imagem

É muito comum que a saída de um VLM ou LLM seja um markdown. Em aplicações reais, é necessário tratar a saída para obter apenas o JSON de fato. Existem muitas bibliotecas que tratam isso de forma transparente. 

In [ ]:
resposta = perguntar_imagem(
    'imagens/03/nota-fiscal.png',
    prompt
)

Markdown(resposta)

## 3. Processando Documento Complexo

Agora vamos processar um documento mais complexo. Vamos usar o mesmo *schema* do notebook anterior.

In [ ]:
img_bgr = cv2.imread('imagens/03/modelo-nfe.jpg')
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12, 8))
plt.imshow(img_rgb)
plt.axis('off')
plt.show()

### 3.1. Definindo schema

In [ ]:
schema = {
    "type": "object",
    "properties": {

        "nota_fiscal": {
            "type": "object",
            "properties": {

                "numero": {
                    "type": "string",
                    "description": "Número da NF-e"
                },

                "serie": {
                    "type": "string"
                },

                "chave_acesso": {
                    "type": "string"
                },

                "data_emissao": {
                    "type": "string"
                }
            }
        },

        "emitente": {
            "type": "object",
            "properties": {

                "razao_social": {
                    "type": "string"
                },

                "cnpj": {
                    "type": "string"
                }
            }
        },

        "destinatario": {
            "type": "object",
            "properties": {

                "razao_social": {
                    "type": "string"
                },

                "cnpj": {
                    "type": "string"
                },

                "cidade": {
                    "type": "string"
                },

                "uf": {
                    "type": "string"
                }
            }
        },

        "itens": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {

                    "codigo": {
                        "type": "string"
                    },

                    "descricao": {
                        "type": "string"
                    },

                    "quantidade": {
                        "type": "number"
                    },

                    "valor_unitario": {
                        "type": "number"
                    },

                    "valor_total": {
                        "type": "number"
                    }
                }
            }
        },

        "totais": {
            "type": "object",
            "properties": {

                "valor_produtos": {
                    "type": "number"
                },

                "valor_frete": {
                    "type": "number"
                },

                "valor_nota": {
                    "type": "number"
                }
            }
        }
    }
}

### 3.2. Definindo prompt

Exatamente o mesmo prompt que usamos no outro documento (só mudando o schema de saída).

In [ ]:
prompt = f"""\
Com a nota fiscal fornecida, extraia as seguintes informações em JSON usando a especificação abaixo delimitada entre ###.
###
{json.dumps(schema)}
###

Retorne apenas o JSON.
"""

### 3.3. Processando imagem

Em alguns casos, vão poder observar que:

- A **chave de acesso** não virá corretamente
- A **cidade** também costuma vir errado
- Os **nomes dos itens** parecem corretos, MAS divergem do que consta na imagem (em alguns casos, isso é bom, mas em outros não)

In [ ]:
resposta = perguntar_imagem(
    'imagens/03/modelo-nfe.jpg',
    prompt
)

Markdown(resposta)

### 3.4. O que fazer quando isso ocorre?

Existem algumas estratégias que podem ser aplicadas para aumentar a acurácia:

1. Usar um VLM mais poderoso, o que pode implicar em custos maiores.
2. Aumentar resolução da imagem
3. Refinar prompt para se atentar a alguns pontos (exemplo: no schema, o campo cidade pode ter uma descrição mencionando que também pode aparecer como município)
4. Usar principios de multi-agente para processar e outro validar informações. Caso haja divergência, processa novamente retroalimentando a observação/ crítica ao prompt inicial.
5. Roteamento de modelo para definir qual modelo é melhor para cada situação.

Todas elas são eficientes e podem ser combinadas.

### 3.5. OCR antes do VLM/ LLM

Uma outra estratégia seria de **obter os dados de OCR em formato markdown (usando IDP) e processando os dados poderiormente com LLM's**.

Isso traz algumas vantagens:

1. Normalmente os OCR's possuem extração de caracteres melhor que os VLM's (melhor acurácia).
2. OCR's são mais baratos
3. Como o texto já foi obtido, LLM's mais simples performam muito bem, o que diminui o custo do processamento.

In [ ]:
# Vamos usar o markdown obtido pelo LandingAI
markdown = Path('imagens/03/modelo-nfe.md').read_text()

# O prompt agora deve embutir o conteúdo do markdown
prompt = f"""\
Com a nota fiscal fornecida, extraia as seguintes informações em JSON usando a especificação abaixo delimitada entre ###.
--- NOTA FISCAL ---
{markdown}
---

###
{json.dumps(schema)}
###

Retorne apenas o JSON.
"""

Agora vão notar que os dados vieram conforme constam na imagem.

In [ ]:
resposta = perguntar(prompt)

Markdown(resposta)

#### Observação

Você deve estar se perguntando.
> O LandingAI já possui métodos para extração, por que devo usar um LLM?

O LandingAI possui de fato um método para extração, porém nem toda solução de OCR/ IDP possui. Essa seria uma ótima combinação em aplicações reais.

## 4. Obtendo informações contextualizadas

Até agora não vimos diferença entre IDP e VLM, salvo a facilidade. VLM's permitem extrair informações mais contextualizadas. Coisa que nem OCR, nem IDP permitem.

> Uma imagem vale mais que 1000 palavras.

### 4.1. Desenhos técnicos

In [ ]:
img_bgr = cv2.imread('imagens/03/hidraulica.webp')
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12, 8))
plt.imshow(img_rgb)
plt.axis('off')
plt.show()

In [ ]:
resposta = perguntar_imagem(
    'imagens/03/hidraulica.webp',
    "Qual o diâmetro da tubulação que vai até o lavatório?"
)

Markdown(resposta)

### 4.2. Gráficos

In [ ]:
img_bgr = cv2.imread('imagens/03/grafico.jpg')
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12, 8))
plt.imshow(img_rgb)
plt.axis('off')
plt.show()

In [ ]:
resposta = perguntar_imagem(
    'imagens/03/grafico.jpg',
    "Retorne uma tabela com os países e sua produtividade (colunas 'País', 'Produtividade'). Retorne apenas a tabela."
)

Markdown(resposta)

## 5. Cuidado com as Alucinações

Apesar dos resultados surpreendentes, ainda é necessário avaliar criteriosamente quando precisará ter validação humana (HITL), pois dependendo da tarefa, o modelo poderá alucinar.

In [ ]:
img_bgr = cv2.imread('imagens/03/sete-erros.jpg')
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12, 8))
plt.imshow(img_rgb)
plt.axis('off')
plt.show()

In [ ]:
resposta = perguntar_imagem(
    'imagens/03/sete-erros.jpg',
    "Indique a localização dos 7 erros."
)

Markdown(resposta)

Normalmente os primeiros erros ele acerta. Da metade em diante, costuma errar.

E isso é algo para crianças...